# Siren vs. Non-Siren Sound Classification Using a CNN

## Problem statement
Build a small Convolutional Neural Network (CNN) that classifies short urban audio recordings into two categories:

- **Siren (1)**
- **Non-siren (0)**

A CNN normally learns from images. Here, we first convert each audio clip into a **mel-spectrogram**—an image-like representation of sound over time and frequency—then train the CNN on these images.

**Dataset:** [UrbanSound8K](https://urbansounddataset.weebly.com/urbansound8k.html), loaded with [Soundata](https://github.com/soundata/soundata). Its official labels include `siren` as one of 10 classes.

**Run this notebook in Google Colab.** The dataset download is large and can take time. The notebook uses a small balanced subset so the model is easier to train and understand. It does not include the dataset files.

## 1. Install the libraries

Run this cell once. If Colab asks to restart the runtime after installation, do so and then continue from the imports.

In [ ]:
!pip -q install soundata librosa scikit-learn matplotlib pandas tensorflow

## 2. Import the libraries

We use pandas to read the dataset labels, librosa to convert audio into spectrograms, and TensorFlow/Keras to build the CNN.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundata
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from tensorflow.keras import layers, models

np.random.seed(42)
tf.random.set_seed(42)

## 3. Download and check UrbanSound8K

Soundata's documented workflow is to initialize the dataset, download it, and validate the expected files. If you already downloaded it to this same environment, Soundata can reuse the local copy.

In [ ]:
missing_files, invalid_files = dataset.validate()

print("Dataset folder:", dataset.data_home)
print("Missing files:", len(missing_files))
print("Files with checksum problems:", len(invalid_files))

if missing_files or invalid_files:
    raise RuntimeError(
        "Dataset validation failed. Fix the download before continuing."
    )

If there are missing files or checksum problems, fix the download before moving on. Do not continue with an incomplete dataset.

## 4. Read the dataset metadata

The CSV tells us the filename, fold, numeric class ID, and text class name. Each recording belongs to one of 10 original sound classes.

In [ ]:
csv_path = os.path.join(
    dataset.data_home, "metadata", "UrbanSound8K.csv"
)

metadata = pd.read_csv(csv_path)

print("Number of recordings:", len(metadata))
print("Metadata columns:", metadata.columns.tolist())
metadata.head()

In [ ]:
print("Recordings per original class:")
print(metadata["class"].value_counts().sort_index())

## 5. Create the two labels

UrbanSound8K uses the class name `siren` for siren recordings. We label those as 1 and group all other sound types under non-siren (0).

In [ ]:
metadata["target"] = (metadata["class"] == "siren").astype(int)

print("Non-siren recordings:", (metadata["target"] == 0).sum())
print("Siren recordings:", (metadata["target"] == 1).sum())

## 6. Create train, validation, and test sets

The dataset is already organized into 10 folds. We use folds 1–9 to choose our training and validation examples, and keep fold 10 aside for testing. This avoids randomly mixing the same fold into both development and test data.

For a shorter run, this notebook uses 400 examples per class for development. The test subset is balanced using the smaller class count in fold 10.

### 6a. Separate the folds

We keep fold 10 for the final test set. The other nine folds are used to prepare the training and validation data.

In [ ]:
train_pool = metadata[metadata["fold"] != 10]
test_pool = metadata[metadata["fold"] == 10]

print("Available training-pool clips:", len(train_pool))
print("Available test-fold clips:", len(test_pool))

### 6b. Sample the training examples

We use the same number of siren and non-siren clips to keep the training data balanced. If the dataset contains fewer examples than requested, the code automatically uses the smaller available count.

In [ ]:
SAMPLES_PER_CLASS = 400

available_train = int(train_pool["target"].value_counts().min())
train_count = min(SAMPLES_PER_CLASS, available_train)

siren_train = train_pool[train_pool["target"] == 1].sample(
    n=train_count, random_state=42
)
other_train = train_pool[train_pool["target"] == 0].sample(
    n=train_count, random_state=42
)

train_data = pd.concat([siren_train, other_train])
train_data = train_data.sample(frac=1, random_state=42).reset_index(drop=True)

print("Sampled training pool:", len(train_data))

### 6c. Create the training and validation sets

The validation set helps us check how the model is doing while it learns. It is kept separate from the training examples.

In [ ]:
train_data, val_data = train_test_split(
    train_data,
    test_size=0.2,
    random_state=42,
    stratify=train_data["target"]
)

print("Training examples:", len(train_data))
print("Validation examples:", len(val_data))

### 6d. Prepare a balanced test set

Fold 10 is not used during training or validation. We use an equal number of siren and non-siren recordings from this fold for testing.

In [ ]:
available_test = int(test_pool["target"].value_counts().min())

if available_test == 0:
    raise ValueError("Fold 10 must contain both siren and non-siren examples.")

siren_test = test_pool[test_pool["target"] == 1].sample(
    n=available_test, random_state=42
)
other_test = test_pool[test_pool["target"] == 0].sample(
    n=available_test, random_state=42
)

test_data = pd.concat([siren_test, other_test])
test_data = test_data.sample(frac=1, random_state=42).reset_index(drop=True)

print("Test examples:", len(test_data))

## 7. Set the audio and spectrogram sizes

All clips are converted to mono audio at the same sample rate and made exactly 4 seconds long. This makes every spectrogram the same size, which is required to train a CNN in batches.

In [ ]:
SAMPLE_RATE = 22050
AUDIO_SECONDS = 4
SAMPLES = SAMPLE_RATE * AUDIO_SECONDS
N_MELS = 64
HOP_LENGTH = 512

## 8. Convert one audio clip into a spectrogram

The Soundata clip's `.audio` property returns the audio signal and its sample rate. We resample it, pad or trim to 4 seconds, then convert it to a mel-spectrogram.

In [ ]:
def audio_to_spectrogram(clip_id):
    audio, original_rate = dataset.clip(clip_id).audio

    if original_rate != SAMPLE_RATE:
        audio = librosa.resample(
            audio,
            orig_sr=original_rate,
            target_sr=SAMPLE_RATE
        )

    audio = librosa.util.fix_length(audio, size=SAMPLES)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SAMPLE_RATE,
        n_fft=1024,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db

## 9. View an example spectrogram

This lets us see the kind of image that the CNN will learn from.

In [ ]:
example_row = train_data.iloc[0]
example_clip_id = example_row["slice_file_name"].replace(".wav", "")

example_spectrogram = audio_to_spectrogram(example_clip_id)

plt.figure(figsize=(8, 3))
librosa.display.specshow(
    example_spectrogram,
    sr=SAMPLE_RATE,
    hop_length=HOP_LENGTH,
    x_axis="time",
    y_axis="mel"
)
plt.title("Example sound spectrogram")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()

print("Actual class:", "siren" if example_row["target"] == 1 else "non-siren")

## 10. Prepare the features and labels

This step reads each selected audio file and converts it into a spectrogram. If an individual file cannot be read, it prints a warning and skips that file.

In [ ]:
def prepare_data(dataframe):
    features = []
    labels = []

    for _, row in dataframe.iterrows():
        clip_id = row["slice_file_name"].replace(".wav", "")

        try:
            spectrogram = audio_to_spectrogram(clip_id)
            features.append(spectrogram)
            labels.append(row["target"])
        except Exception as error:
            print("Could not read", row["slice_file_name"], "-", error)

    return np.array(features, dtype=np.float32), np.array(labels, dtype=np.float32)

Convert the three data splits. This may take a few minutes because every selected audio recording must be processed.

In [ ]:
X_train, y_train = prepare_data(train_data)
X_val, y_val = prepare_data(val_data)
X_test, y_test = prepare_data(test_data)

print("Training feature shape:", X_train.shape)
print("Validation feature shape:", X_val.shape)
print("Test feature shape:", X_test.shape)

## 11. Scale the spectrograms

The spectrogram is measured in decibels. We scale the values to a range near 0–1 and add a final channel dimension because CNN image inputs have height, width, and channels.

In [ ]:
def scale_features(features):
    features = ((features + 80) / 80).clip(0, 1)
    return features[..., np.newaxis]

X_train = scale_features(X_train)
X_val = scale_features(X_val)
X_test = scale_features(X_test)

print("CNN input shape:", X_train.shape[1:])

## 12. Build the CNN

- `Conv2D` learns patterns in the spectrogram.
- `MaxPooling2D` reduces the size of the learned feature maps.
- `Flatten` turns the maps into a vector.
- The final sigmoid unit outputs a probability between 0 and 1.

In [ ]:
model = models.Sequential([
    layers.Input(shape=X_train.shape[1:]),

    layers.Conv2D(8, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 13. Train the model

The model learns from the training examples and checks its performance on the validation set after each epoch. The test data is held back until evaluation.

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=8,
    batch_size=16
)

## 14. Plot training results

If training accuracy keeps rising while validation accuracy stops improving, the model may be overfitting.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

## 15. Evaluate on the test set

Accuracy is the percentage of correct predictions. Precision and recall help us understand how often siren predictions are correct and how many siren clips the model finds.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(test_accuracy, 4))

In [ ]:
probabilities = model.predict(X_test, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)

print(classification_report(
    y_test.astype(int),
    predictions,
    labels=[0, 1],
    target_names=["non-siren", "siren"],
    zero_division=0
))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test.astype(int),
    predictions,
    labels=[0, 1],
    display_labels=["non-siren", "siren"],
    values_format="d"
)
plt.title("Test confusion matrix")
plt.show()

## 16. Try one example prediction

The probability is the model's estimated chance that the clip contains a siren. A value of 0.5 or higher is classified as siren.

In [ ]:
index = 0

siren_probability = float(
    model.predict(X_test[index:index + 1], verbose=0)[0][0]
)

actual_class = "siren" if y_test[index] == 1 else "non-siren"
predicted_class = "siren" if siren_probability >= 0.5 else "non-siren"

print("Actual class:", actual_class)
print("Predicted class:", predicted_class)
print("Siren probability:", round(siren_probability, 3))

## Conclusion

This project demonstrates a basic audio deep-learning workflow: metadata inspection, binary labels, audio-to-spectrogram conversion, CNN training, and test evaluation.

**Do not report an accuracy until you run the notebook.** The results depend on the data subset and training run. The notebook has been organized for readability, but the full download and model training still need to be run in your Colab environment.

Dataset reference: Salamon, Jacoby, and Bello, “A Dataset and Taxonomy for Urban Sound Research,” ACM Multimedia, 2014. Read the [official UrbanSound8K terms](https://urbansounddataset.weebly.com/urbansound8k.html) before use.